# Chapter 5.4: Contrastive Learning for Sequential Recommendation

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain why **contrastive learning** helps sequential recommendation (data sparsity, representation quality)
2. Implement **CL4SRec** with multiple data augmentation strategies and InfoNCE loss
3. Understand **CoSeRec** and its contrastive self-supervised pre-training approach
4. Implement **DuoRec** with model-level augmentation (dropout-based views)
5. Design and compare augmentation strategies: crop, mask, reorder, substitute, insert
6. Build a self-supervised pre-training followed by fine-tuning pipeline
7. Analyze the effect of contrastive weight ($\lambda$) and temperature ($\tau$) on performance

## Prerequisites

- Chapter 5.2 (SASRec architecture)
- Familiarity with contrastive learning (SimCLR, InfoNCE)
- PyTorch training loops

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USERNAME/rec_system/blob/main/notebooks/part5/chapter_5.4_contrastive_seq.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/USERNAME/rec_system/main/notebooks/part5/chapter_5.4_contrastive_seq.ipynb)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import math
import random
import copy

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cpu")
print(f"Using device: {device}")

## 1. Why Contrastive Learning for Sequential Rec?

Sequential recommendation suffers from **data sparsity**: each user has limited interactions. Contrastive learning helps by:

1. **Augmenting data**: Creating multiple views of the same sequence
2. **Better representations**: Encouraging similar sequences to have similar embeddings
3. **Regularization**: Preventing overfitting to sparse supervision signals

The general framework:

$$\mathcal{L} = \mathcal{L}_{\text{rec}} + \lambda \cdot \mathcal{L}_{\text{CL}}$$

where $\mathcal{L}_{\text{rec}}$ is the recommendation loss (cross-entropy/BPR) and $\mathcal{L}_{\text{CL}}$ is the contrastive loss (InfoNCE).

> **💡 Concept:** The core idea is that two augmented views of the same sequence should produce similar representations (positives), while views from different sequences should be dissimilar (negatives).

## 2. Synthetic Data

In [ ]:
def generate_sequences(n_users=1000, n_items=500, min_len=5, max_len=50, seed=42):
    rng = np.random.RandomState(seed)
    n_categories = 20
    item_category = rng.randint(0, n_categories, size=n_items)
    item_popularity = rng.power(0.5, size=n_items)
    item_popularity /= item_popularity.sum()
    
    sequences = []
    for u in range(n_users):
        seq_len = rng.randint(min_len, max_len + 1)
        user_cats = rng.choice(n_categories, size=3, replace=False)
        seq = []
        for _ in range(seq_len):
            if rng.random() < 0.6:
                cat = rng.choice(user_cats)
                cat_items = np.where(item_category == cat)[0]
                if len(cat_items) > 0:
                    probs = item_popularity[cat_items]
                    probs /= probs.sum()
                    item = rng.choice(cat_items, p=probs)
                else:
                    item = rng.choice(n_items, p=item_popularity)
            else:
                item = rng.choice(n_items, p=item_popularity)
            seq.append(int(item))
        sequences.append(seq)
    return sequences, item_category, item_popularity

N_ITEMS = 500
MAX_SEQ_LEN = 50
sequences, item_category, item_popularity = generate_sequences(n_users=1000, n_items=N_ITEMS)

train_seqs = sequences[:800]
val_seqs = sequences[800:900]
test_seqs = sequences[900:]

print(f"Train: {len(train_seqs)}, Val: {len(val_seqs)}, Test: {len(test_seqs)}")

## 3. Augmentation Strategies

Five main augmentation strategies used in CL4SRec and CoSeRec:

1. **Crop**: Random contiguous subsequence
2. **Mask**: Randomly replace items with a mask token
3. **Reorder**: Shuffle a contiguous segment
4. **Substitute**: Replace items with similar items (based on co-occurrence)
5. **Insert**: Insert random items into the sequence

In [ ]:
class SequenceAugmentor:
    """Collection of sequence augmentation strategies for contrastive learning."""
    
    def __init__(self, n_items, item_similarity=None):
        self.n_items = n_items
        self.item_similarity = item_similarity  # For substitute augmentation
    
    def crop(self, seq, ratio=0.6):
        """Random contiguous subsequence."""
        crop_len = max(2, int(len(seq) * ratio))
        start = random.randint(0, max(0, len(seq) - crop_len))
        return seq[start:start + crop_len]
    
    def mask(self, seq, ratio=0.2, mask_value=0):
        """Randomly mask items."""
        aug = seq.copy()
        n_mask = max(1, int(len(aug) * ratio))
        positions = random.sample(range(len(aug)), min(n_mask, len(aug)))
        for pos in positions:
            aug[pos] = mask_value
        return aug
    
    def reorder(self, seq, ratio=0.2):
        """Shuffle a random contiguous segment."""
        aug = seq.copy()
        seg_len = max(2, int(len(aug) * ratio))
        start = random.randint(0, max(0, len(aug) - seg_len))
        segment = aug[start:start + seg_len]
        random.shuffle(segment)
        aug[start:start + seg_len] = segment
        return aug
    
    def substitute(self, seq, ratio=0.2):
        """Replace items with random items (or similar items if available)."""
        aug = seq.copy()
        n_sub = max(1, int(len(aug) * ratio))
        positions = random.sample(range(len(aug)), min(n_sub, len(aug)))
        for pos in positions:
            aug[pos] = random.randint(0, self.n_items - 1)
        return aug
    
    def insert(self, seq, ratio=0.2):
        """Insert random items at random positions."""
        aug = seq.copy()
        n_insert = max(1, int(len(aug) * ratio))
        for _ in range(n_insert):
            pos = random.randint(0, len(aug))
            item = random.randint(0, self.n_items - 1)
            aug.insert(pos, item)
        return aug
    
    def augment(self, seq, strategy="crop", **kwargs):
        """Apply a named augmentation strategy."""
        fn = getattr(self, strategy)
        return fn(seq, **kwargs)

# Demo
augmentor = SequenceAugmentor(N_ITEMS)
demo_seq = list(range(10))
print(f"Original:    {demo_seq}")
print(f"Crop:        {augmentor.crop(demo_seq, 0.6)}")
print(f"Mask:        {augmentor.mask(demo_seq, 0.3)}")
print(f"Reorder:     {augmentor.reorder(demo_seq, 0.3)}")
print(f"Substitute:  {augmentor.substitute(demo_seq, 0.3)}")
print(f"Insert:      {augmentor.insert(demo_seq, 0.2)}")

## 4. InfoNCE Loss

The **InfoNCE** (Noise-Contrastive Estimation) loss used in contrastive learning:

$$\mathcal{L}_{\text{InfoNCE}} = -\log \frac{\exp(\text{sim}(\mathbf{z}_i, \mathbf{z}_i^+) / \tau)}{\sum_{j=1}^{N} \exp(\text{sim}(\mathbf{z}_i, \mathbf{z}_j) / \tau)}$$

where:
- $\mathbf{z}_i, \mathbf{z}_i^+$ are two augmented views of the same sequence (positive pair)
- $\mathbf{z}_j$ includes all other sequences in the batch (negatives)
- $\tau$ is the temperature (controls sharpness)
- $\text{sim}(\cdot, \cdot)$ is cosine similarity

In [ ]:
def info_nce_loss(z1, z2, temperature=0.1):
    """
    InfoNCE contrastive loss.
    z1, z2: (batch_size, embed_dim) — two views of the same sequences
    Uses in-batch negatives.
    """
    batch_size = z1.size(0)
    
    # L2 normalize
    z1 = F.normalize(z1, dim=-1)
    z2 = F.normalize(z2, dim=-1)
    
    # Cosine similarity matrix
    sim_matrix = z1 @ z2.T / temperature  # (batch, batch)
    
    # Positive pairs are on the diagonal
    labels = torch.arange(batch_size, device=z1.device)
    
    # Symmetric loss
    loss_12 = F.cross_entropy(sim_matrix, labels)
    loss_21 = F.cross_entropy(sim_matrix.T, labels)
    
    return (loss_12 + loss_21) / 2

# Demo
z1 = torch.randn(8, 64)
z2 = z1 + torch.randn(8, 64) * 0.1  # Similar views
z3 = torch.randn(8, 64)  # Random (dissimilar)

print(f"InfoNCE (similar views):   {info_nce_loss(z1, z2, temperature=0.1):.4f}")
print(f"InfoNCE (random views):    {info_nce_loss(z1, z3, temperature=0.1):.4f}")

## 5. CL4SRec: Contrastive Learning for Sequential Recommendation

**CL4SRec** (Xie et al., ICDE 2022) combines SASRec with contrastive self-supervised learning:

1. Apply two random augmentations to each sequence to create two views
2. Encode both views with the same SASRec backbone
3. Compute InfoNCE loss between the two views
4. Joint training: $\mathcal{L} = \mathcal{L}_{\text{CE}} + \lambda \mathcal{L}_{\text{InfoNCE}}$

> **💡 Concept:** CL4SRec uses data-level augmentation: the same model processes different augmented inputs. This is similar to SimCLR in computer vision.

In [ ]:
class SASRecBackbone(nn.Module):
    """SASRec backbone for CL4SRec."""
    
    def __init__(self, n_items, max_len=50, embed_dim=64, n_heads=2, n_layers=2, dropout=0.2):
        super().__init__()
        self.n_items = n_items
        self.max_len = max_len
        self.embed_dim = embed_dim
        
        self.item_emb = nn.Embedding(n_items + 1, embed_dim, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, embed_dim)
        self.emb_dropout = nn.Dropout(dropout)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads,
            dim_feedforward=embed_dim * 4, dropout=dropout,
            activation="gelu", batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.final_norm = nn.LayerNorm(embed_dim)
        
        nn.init.xavier_uniform_(self.item_emb.weight[1:])
        nn.init.xavier_uniform_(self.pos_emb.weight)
    
    def encode(self, item_seq):
        """Encode a sequence, return hidden states."""
        seq_len = item_seq.size(1)
        positions = torch.arange(seq_len, device=item_seq.device).unsqueeze(0)
        x = self.item_emb(item_seq) + self.pos_emb(positions)
        x = self.emb_dropout(x)
        
        causal_mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool().to(item_seq.device)
        padding_mask = (item_seq == 0)
        
        x = self.transformer(x, mask=causal_mask, src_key_padding_mask=padding_mask)
        return self.final_norm(x)
    
    def get_sequence_repr(self, item_seq):
        """Get sequence-level representation (last hidden state)."""
        hidden = self.encode(item_seq)  # (batch, seq_len, embed_dim)
        return hidden[:, -1, :]  # (batch, embed_dim)
    
    def predict(self, item_seq):
        """Predict next-item scores."""
        repr_ = self.get_sequence_repr(item_seq)
        item_embs = self.item_emb.weight[1:self.n_items + 1]
        return repr_ @ item_embs.T

class CL4SRec(nn.Module):
    """CL4SRec: Contrastive Learning for Sequential Recommendation (Xie et al., 2022)."""
    
    def __init__(self, n_items, max_len=50, embed_dim=64, n_heads=2, n_layers=2,
                 dropout=0.2, cl_weight=0.1, temperature=0.1,
                 aug1="crop", aug2="mask"):
        super().__init__()
        self.backbone = SASRecBackbone(
            n_items, max_len, embed_dim, n_heads, n_layers, dropout
        )
        self.cl_weight = cl_weight
        self.temperature = temperature
        self.aug1 = aug1
        self.aug2 = aug2
        self.n_items = n_items
        self.max_len = max_len
    
    def predict(self, item_seq):
        return self.backbone.predict(item_seq)
    
    def compute_cl_loss(self, seq_repr1, seq_repr2):
        """Compute InfoNCE contrastive loss between two views."""
        return info_nce_loss(seq_repr1, seq_repr2, self.temperature)

print("CL4SRec model created successfully.")

In [ ]:
class CL4SRecDataset(Dataset):
    """Dataset for CL4SRec with on-the-fly augmentation."""
    
    def __init__(self, sequences, n_items, max_len=50, aug1="crop", aug2="mask"):
        self.sequences = sequences
        self.max_len = max_len
        self.augmentor = SequenceAugmentor(n_items)
        self.aug1 = aug1
        self.aug2 = aug2
    
    def _pad_and_convert(self, seq):
        """Left-pad and 1-index a sequence."""
        seq = seq[-self.max_len:]
        padded = [0] * (self.max_len - len(seq)) + [x + 1 for x in seq]
        return torch.tensor(padded, dtype=torch.long)
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        seq = self.sequences[idx]
        inp = seq[:-1]
        target = seq[-1]
        
        # Create two augmented views
        view1 = self.augmentor.augment(inp, self.aug1)
        view2 = self.augmentor.augment(inp, self.aug2)
        
        return (
            self._pad_and_convert(inp),     # Original
            self._pad_and_convert(view1),   # Augmented view 1
            self._pad_and_convert(view2),   # Augmented view 2
            torch.tensor(target, dtype=torch.long),
        )

cl_train_ds = CL4SRecDataset(train_seqs, N_ITEMS, MAX_SEQ_LEN, aug1="crop", aug2="mask")
cl_train_loader = DataLoader(cl_train_ds, batch_size=128, shuffle=True)

# Show a sample
orig, v1, v2, target = cl_train_ds[0]
print(f"Original (last 10):  {orig[-10:].tolist()}")
print(f"View 1 (last 10):    {v1[-10:].tolist()}")
print(f"View 2 (last 10):    {v2[-10:].tolist()}")
print(f"Target: {target.item()}")

## 6. Training CL4SRec

In [ ]:
def train_cl4srec(model, train_loader, val_ds, n_epochs=15, lr=0.001):
    """Train CL4SRec with joint recommendation + contrastive loss."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.98))
    rec_criterion = nn.CrossEntropyLoss()
    
    history = {"rec_loss": [], "cl_loss": [], "total_loss": [], "val_hr10": []}
    
    # Simple val loader (no augmentation needed)
    val_pairs = []
    for seq in val_seqs:
        inp = seq[:-1][-MAX_SEQ_LEN:]
        padded = [0] * (MAX_SEQ_LEN - len(inp)) + [x + 1 for x in inp]
        val_pairs.append((torch.tensor(padded, dtype=torch.long), seq[-1]))
    
    for epoch in range(n_epochs):
        model.train()
        total_rec, total_cl, total_all = 0, 0, 0
        n_batches = 0
        
        for orig, view1, view2, target in train_loader:
            # Recommendation loss on original sequence
            logits = model.predict(orig)
            rec_loss = rec_criterion(logits, target)
            
            # Contrastive loss between augmented views
            repr1 = model.backbone.get_sequence_repr(view1)
            repr2 = model.backbone.get_sequence_repr(view2)
            cl_loss = model.compute_cl_loss(repr1, repr2)
            
            loss = rec_loss + model.cl_weight * cl_loss
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            
            total_rec += rec_loss.item()
            total_cl += cl_loss.item()
            total_all += loss.item()
            n_batches += 1
        
        history["rec_loss"].append(total_rec / n_batches)
        history["cl_loss"].append(total_cl / n_batches)
        history["total_loss"].append(total_all / n_batches)
        
        # Validation
        model.eval()
        hits = 0
        with torch.no_grad():
            for i in range(0, len(val_pairs), 128):
                batch_items = torch.stack([p[0] for p in val_pairs[i:i+128]])
                batch_targets = [p[1] for p in val_pairs[i:i+128]]
                logits = model.predict(batch_items)
                _, topk = logits.topk(10, dim=-1)
                for j, t in enumerate(batch_targets):
                    if t in topk[j].tolist():
                        hits += 1
        hr10 = hits / len(val_pairs)
        history["val_hr10"].append(hr10)
        
        if (epoch + 1) % 3 == 0:
            print(f"Epoch {epoch+1}/{n_epochs} — Rec: {history['rec_loss'][-1]:.4f}, "
                  f"CL: {history['cl_loss'][-1]:.4f}, Val HR@10: {hr10:.4f}")
    
    return history

torch.manual_seed(SEED)
cl4srec = CL4SRec(n_items=N_ITEMS, max_len=MAX_SEQ_LEN, embed_dim=64,
                  cl_weight=0.1, temperature=0.1, aug1="crop", aug2="mask")
cl_history = train_cl4srec(cl4srec, cl_train_loader, val_seqs, n_epochs=15)

In [ ]:
# Visualize training
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

epochs = range(1, len(cl_history["rec_loss"]) + 1)

axes[0].plot(epochs, cl_history["rec_loss"], marker="o", label="Rec Loss", color="steelblue")
axes[0].plot(epochs, cl_history["cl_loss"], marker="s", label="CL Loss", color="coral")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("CL4SRec: Loss Components")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, cl_history["total_loss"], marker="o", color="purple")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Total Loss")
axes[1].set_title("CL4SRec: Total Loss")
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs, cl_history["val_hr10"], marker="s", color="green")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("HR@10")
axes[2].set_title("CL4SRec: Validation HR@10")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. DuoRec: Model-Level Augmentation

**DuoRec** (Qiu et al., WSDM 2022) uses **model-level augmentation** instead of data-level:

- Pass the **same input** through the model twice with **different dropout masks**
- The two forward passes produce different representations (due to dropout)
- These become the two contrastive views

Advantages:
- No need to design augmentation strategies
- Preserves sequence integrity (no information loss from masking/cropping)
- Computationally simpler

> **🔑 Pro Tip:** DuoRec shows that model-level augmentation (dropout noise) can be as effective as data-level augmentation, while being simpler to implement.

In [ ]:
class DuoRec(nn.Module):
    """DuoRec: Contrastive learning with model-level augmentation (Qiu et al., 2022)."""
    
    def __init__(self, n_items, max_len=50, embed_dim=64, n_heads=2, n_layers=2,
                 dropout=0.3, cl_weight=0.1, temperature=0.1):
        super().__init__()
        self.backbone = SASRecBackbone(
            n_items, max_len, embed_dim, n_heads, n_layers, dropout
        )
        self.cl_weight = cl_weight
        self.temperature = temperature
        self.n_items = n_items
    
    def predict(self, item_seq):
        return self.backbone.predict(item_seq)
    
    def get_contrastive_views(self, item_seq):
        """
        Get two different views by forward-passing through the model twice
        (dropout creates different representations).
        Must be called in training mode!
        """
        assert self.training, "Must be in training mode for dropout augmentation"
        repr1 = self.backbone.get_sequence_repr(item_seq)
        repr2 = self.backbone.get_sequence_repr(item_seq)  # Different dropout mask!
        return repr1, repr2

# Quick test: verify that two forward passes give different results in train mode
duorec = DuoRec(n_items=N_ITEMS, max_len=MAX_SEQ_LEN, dropout=0.3)
duorec.train()
test_input = torch.randint(1, N_ITEMS, (4, MAX_SEQ_LEN))
r1, r2 = duorec.get_contrastive_views(test_input)
cosine_sim = F.cosine_similarity(r1, r2, dim=-1)
print(f"Cosine similarity between two views: {cosine_sim.detach().numpy()}")
print(f"(Should be close to but not exactly 1.0 due to dropout)")

## 8. Comparing Augmentation Strategies

Let's systematically compare different augmentation combinations.

In [ ]:
def quick_train_evaluate(aug1, aug2, n_epochs=10):
    """Quick train/eval for an augmentation pair."""
    torch.manual_seed(SEED)
    ds = CL4SRecDataset(train_seqs, N_ITEMS, MAX_SEQ_LEN, aug1=aug1, aug2=aug2)
    loader = DataLoader(ds, batch_size=128, shuffle=True)
    
    model = CL4SRec(n_items=N_ITEMS, max_len=MAX_SEQ_LEN, embed_dim=64,
                    cl_weight=0.1, temperature=0.1, aug1=aug1, aug2=aug2)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(n_epochs):
        model.train()
        for orig, v1, v2, target in loader:
            logits = model.predict(orig)
            rec_loss = criterion(logits, target)
            r1 = model.backbone.get_sequence_repr(v1)
            r2 = model.backbone.get_sequence_repr(v2)
            cl_loss = model.compute_cl_loss(r1, r2)
            loss = rec_loss + 0.1 * cl_loss
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
    
    # Evaluate
    model.eval()
    hits = 0
    total = 0
    with torch.no_grad():
        for seq in val_seqs:
            inp = seq[:-1][-MAX_SEQ_LEN:]
            padded = [0] * (MAX_SEQ_LEN - len(inp)) + [x + 1 for x in inp]
            item_seq = torch.tensor(padded, dtype=torch.long).unsqueeze(0)
            logits = model.predict(item_seq)
            _, topk = logits.topk(10, dim=-1)
            if seq[-1] in topk[0].tolist():
                hits += 1
            total += 1
    return hits / total

# Compare augmentation pairs
aug_pairs = [
    ("crop", "crop"),
    ("crop", "mask"),
    ("crop", "reorder"),
    ("mask", "mask"),
    ("mask", "reorder"),
    ("reorder", "substitute"),
]

results = {}
for a1, a2 in aug_pairs:
    hr10 = quick_train_evaluate(a1, a2, n_epochs=8)
    results[f"{a1}+{a2}"] = hr10
    print(f"{a1:>10s} + {a2:<10s}: HR@10 = {hr10:.4f}")

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(10, 5))
names = list(results.keys())
values = list(results.values())
colors = plt.cm.Set2(np.linspace(0, 1, len(names)))

bars = ax.barh(names, values, color=colors, edgecolor="black")
ax.set_xlabel("HR@10")
ax.set_title("CL4SRec: Augmentation Strategy Comparison")
for bar, val in zip(bars, values):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f"{val:.4f}", va="center", fontsize=10)
plt.tight_layout()
plt.show()

## 9. Temperature and CL Weight Sensitivity

> **⚠️ Common Pitfall:** The temperature $\tau$ and contrastive weight $\lambda$ are critical hyperparameters. Too low $\tau$ makes the loss too sharp (focuses only on hardest negatives); too high $\lambda$ can overwhelm the recommendation objective.

In [ ]:
# Analyze temperature effect on InfoNCE
z1 = F.normalize(torch.randn(32, 64), dim=-1)
z2 = F.normalize(z1 + torch.randn(32, 64) * 0.3, dim=-1)

temperatures = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0]
losses = [info_nce_loss(z1, z2, t).item() for t in temperatures]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(temperatures, losses, marker="o", color="steelblue")
ax.set_xlabel("Temperature (tau)")
ax.set_ylabel("InfoNCE Loss")
ax.set_title("Effect of Temperature on InfoNCE Loss")
ax.set_xscale("log")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Self-Supervised Pre-Training Pipeline

An alternative to joint training is a **two-stage** approach:
1. **Pre-train** with contrastive loss only (no labels needed)
2. **Fine-tune** on the recommendation task

This is especially useful when labeled data is scarce but unlabeled interaction data is abundant.

In [ ]:
def pretrain_contrastive(model, sequences, n_items, max_len, n_epochs=5, lr=0.001):
    """Pre-train with contrastive loss only."""
    ds = CL4SRecDataset(sequences, n_items, max_len, aug1="crop", aug2="mask")
    loader = DataLoader(ds, batch_size=128, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    for epoch in range(n_epochs):
        model.train()
        total_loss = 0
        n_b = 0
        for _, v1, v2, _ in loader:
            r1 = model.backbone.get_sequence_repr(v1)
            r2 = model.backbone.get_sequence_repr(v2)
            loss = info_nce_loss(r1, r2, temperature=0.1)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_b += 1
        print(f"Pre-train epoch {epoch+1}/{n_epochs} — CL Loss: {total_loss/n_b:.4f}")

def finetune_rec(model, sequences, n_items, max_len, n_epochs=10, lr=0.0005):
    """Fine-tune on recommendation task."""
    class SimpleSeqDS(Dataset):
        def __init__(self, seqs, ml):
            self.seqs = seqs
            self.ml = ml
        def __len__(self):
            return len(self.seqs)
        def __getitem__(self, idx):
            s = self.seqs[idx]
            inp = s[:-1][-self.ml:]
            padded = [0] * (self.ml - len(inp)) + [x + 1 for x in inp]
            return torch.tensor(padded, dtype=torch.long), torch.tensor(s[-1], dtype=torch.long)
    
    ds = SimpleSeqDS(sequences, max_len)
    loader = DataLoader(ds, batch_size=128, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(n_epochs):
        model.train()
        total_loss = 0
        n_b = 0
        for item_seq, target in loader:
            logits = model.predict(item_seq)
            loss = criterion(logits, target)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_b += 1
        if (epoch + 1) % 3 == 0:
            print(f"Fine-tune epoch {epoch+1}/{n_epochs} — Loss: {total_loss/n_b:.4f}")

# Demo: Pre-train + Fine-tune
torch.manual_seed(SEED)
pretrained_model = CL4SRec(n_items=N_ITEMS, max_len=MAX_SEQ_LEN, embed_dim=64)
print("=== Pre-training ===")
pretrain_contrastive(pretrained_model, train_seqs, N_ITEMS, MAX_SEQ_LEN, n_epochs=5)
print("\n=== Fine-tuning ===")
finetune_rec(pretrained_model, train_seqs, N_ITEMS, MAX_SEQ_LEN, n_epochs=10)

## 11. Exercises

### 🏋️ Exercise 1: Implement CL4SRec with All Augmentation Strategies

In [ ]:
# 🏋️ Exercise 1: Enhanced CL4SRec
#
# Implement a version of CL4SRec that randomly selects augmentation strategies
# for each sample (rather than using fixed strategies for the whole dataset).

class CL4SRecRandom(nn.Module):
    """CL4SRec with randomly selected augmentation per sample."""
    
    def __init__(self, n_items, max_len=50, embed_dim=64, n_heads=2, n_layers=2,
                 dropout=0.2, cl_weight=0.1, temperature=0.1):
        super().__init__()
        # TODO: Implement
        # Same as CL4SRec but the dataset randomly picks from
        # ["crop", "mask", "reorder", "substitute"] for each view
        pass

# TODO: Create a dataset class that randomly selects augmentation
# TODO: Train and evaluate, compare with fixed augmentation

### 🏋️ Exercise 2: Implement DuoRec Training Loop

In [ ]:
# 🏋️ Exercise 2: DuoRec Training
#
# Train DuoRec using model-level augmentation (dropout-based views).
# Compare with CL4SRec.

def train_duorec(model, train_seqs, val_seqs, n_items, max_len, n_epochs=15, lr=0.001):
    """Train DuoRec."""
    # TODO: Implement
    # 1. Create simple dataset (no augmentation needed)
    # 2. For each batch:
    #    a. Forward pass for recommendation loss
    #    b. Two forward passes (with different dropout) for CL loss
    #    c. Joint loss = rec_loss + lambda * cl_loss
    # 3. Track and return metrics
    pass

# torch.manual_seed(SEED)
# duorec_model = DuoRec(n_items=N_ITEMS, max_len=MAX_SEQ_LEN)
# duorec_history = train_duorec(duorec_model, train_seqs, val_seqs, N_ITEMS, MAX_SEQ_LEN)

### 🏋️ Exercise 3: Hyperparameter Sensitivity Analysis

In [ ]:
# 🏋️ Exercise 3: Sensitivity to lambda and tau
#
# Run CL4SRec with different contrastive weights and temperatures:
# - lambda: [0.01, 0.05, 0.1, 0.2, 0.5]
# - tau: [0.05, 0.1, 0.2, 0.5, 1.0]
#
# Create a heatmap of HR@10 for different (lambda, tau) combinations.

# TODO: Implement grid search over lambda and tau
# TODO: Create heatmap visualization with plt.imshow or plt.pcolormesh
pass

## Summary

| Method | Year | Augmentation Level | Key Idea |
|--------|------|-------------------|----------|
| CL4SRec | 2022 | Data-level (crop, mask, reorder) | Multiple augmentation + InfoNCE |
| CoSeRec | 2021 | Data-level (robust augmentations) | Contrastive self-supervised pre-training |
| DuoRec | 2022 | Model-level (dropout) | Simpler, no data corruption |

**Key Takeaways:**
1. Contrastive learning improves sequential rec by regularizing representations and combating sparsity
2. CL4SRec combines data augmentation with InfoNCE loss in a joint training framework
3. DuoRec shows that model-level augmentation (dropout noise) can match data-level approaches
4. The contrastive weight $\lambda$ and temperature $\tau$ significantly affect performance
5. Pre-training + fine-tuning is effective when labeled data is scarce
6. Next chapter: **Graph Neural Networks** for recommendation